In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'https://www.pff.com/news/nfl-roster-rankings-all-32-teams-2024-strengths-weaknesses-x-factors'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

response = requests.get(url, headers=headers)
print(f"Status code: {response.status_code}")

if response.status_code != 200:
    raise Exception(f"Failed to retrieve PFF page. Status code: {response.status_code}")

soup = BeautifulSoup(response.content, 'html.parser')

data = []
team_headers = soup.find_all('h3')
tables = soup.find_all('table')

for header, player_table in zip(team_headers, tables):
    team_name = header.text.strip()
    rows = player_table.find_all('tr')
    for row in rows[1:]:
        cells = row.find_all('td')
        if len(cells) == 2:
            for cell in cells:
                cell_text = cell.text.strip()
                if cell_text:
                    parts = cell_text.split(' ')
                    position = parts[0]
                    player_name = ' '.join(parts[1:-1])
                    grade = parts[-1].strip('()')
                    data.append({'Team': team_name, 'Position': position, 'Player': player_name, 'Grade': grade})

currAVs = pd.DataFrame(data)
currAVs['Grade'] = currAVs['Grade'].str.replace('*', '', regex=False)
currAVs['Grade'] = pd.to_numeric(currAVs['Grade'], errors='coerce')

print(f"Scraped {len(currAVs)} player grades across {currAVs['Team'].nunique()} teams")
currAVs.head(10)


In [ ]:
import pandas as pd
import numpy as np

currAVs['Grade'] = pd.to_numeric(currAVs['Grade'], errors='coerce')

def weighted_top_n(grades, weights):
    """Return weighted average of top N grades. weights must sum to 1."""
    sorted_grades = sorted(grades.dropna(), reverse=True)
    n = len(weights)
    if len(sorted_grades) == 0:
        return None
    # Pad with the last grade if fewer players than expected
    while len(sorted_grades) < n:
        sorted_grades.append(sorted_grades[-1])
    return sum(g * w for g, w in zip(sorted_grades[:n], weights))

# Grading rules:
#   QB    - top 1 starter only
#   RB    - weighted top 2 (65/35) to reflect starter dominance
#   WR    - weighted top 3 (50/30/20)
#   TE    - weighted top 2 (65/35)
#   OLine - top 5 equal weight (all starters)
#   DST   - top 7 equal weight (core starters)

grade_rules = {
    'qb':    (['QB'],                   lambda g: weighted_top_n(g, [1.0])),
    'rb':    (['RB'],                   lambda g: weighted_top_n(g, [0.65, 0.35])),
    'wr':    (['WR'],                   lambda g: weighted_top_n(g, [0.50, 0.30, 0.20])),
    'te':    (['TE'],                   lambda g: weighted_top_n(g, [0.65, 0.35])),
    'oline': (['LT', 'LG', 'C', 'RG', 'RT'], lambda g: weighted_top_n(g, [0.2, 0.2, 0.2, 0.2, 0.2])),
    'dst':   (['Edge', 'LB', 'Dl', 'CB', 'S'], lambda g: weighted_top_n(g, [1/7]*7)),
}

corrections = {
    '1. San Francisco 49Ers': 'SF',  '1. San Francisco 49ers': 'SF',
    '2. Kansas City Chiefs': 'KC',   '3. Philadelphia Eagles': 'PHI',
    '4. New York Jets': 'NYJ',       '5. Baltimore Ravens': 'BAL',
    '6. Detroit Lions': 'DET',       '7. Houston Texans': 'HOU',
    '8. Cincinnati Bengals': 'CIN',  '9. Dallas Cowboys': 'DAL',
    '10. Buffalo Bills': 'BUF',      '11. Miami Dolphins': 'MIA',
    '12. Cleveland Browns': 'CLE',   '13. Green Bay Packers': 'GB',
    '14. Los Angeles Rams': 'LA',    '15. Atlanta Falcons': 'ATL',
    '16. Pittsburgh Steelers': 'PIT','17. Seattle Seahawks': 'SEA',
    '18. Tampa Bay Buccaneers': 'TB','19. Jacksonville Jaguars': 'JAX',
    '20. Chicago Bears': 'CHI',      '21. Minnesota Vikings': 'MIN',
    '22. Indianapolis Colts': 'IND', '23. Las Vegas Raiders': 'LV',
    '24. New Orleans Saints': 'NO',  '25. Tennessee Titans': 'TEN',
    '26. Los Angeles Chargers': 'LAC','27. Washington Commanders': 'WAS',
    '28. Arizona Cardinals': 'ARI',  '29. New England Patriots': 'NE',
    '30. Carolina Panthers': 'CAR',  '31. New York Giants': 'NYG',
    '32. Denver Broncos': 'DEN',
}

currAVs['Team'] = currAVs['Team'].str.title().replace(corrections)

new_data = []
for team in currAVs['Team'].unique():
    team_df = currAVs[currAVs['Team'] == team]
    row = {'Team': team}
    for group, (positions, agg_fn) in grade_rules.items():
        grades = team_df[team_df['Position'].isin(positions)]['Grade']
        row[group] = agg_fn(grades)
    new_data.append(row)

currAVs = pd.DataFrame(new_data)

# Combine WR + TE into a single wrte grade (60/40 split)
currAVs['wrte'] = currAVs['wr'].fillna(0) * 0.60 + currAVs['te'].fillna(0) * 0.40
currAVs = currAVs.drop(columns=['wr', 'te'])

# Robust normalization: clip to 10th-90th percentile range before scaling
# so one outlier team can't compress everyone else into a narrow band
columns_to_scale = ['oline', 'qb', 'rb', 'wrte', 'dst']
currAVs_scaled = currAVs.copy()

for col in columns_to_scale:
    vals = currAVs[col].dropna()
    p10 = np.percentile(vals, 10)
    p90 = np.percentile(vals, 90)
    clipped = currAVs[col].clip(lower=p10, upper=p90)
    col_min, col_max = clipped.min(), clipped.max()
    if col_max > col_min:
        currAVs_scaled[col] = (clipped - col_min) / (col_max - col_min)
    else:
        currAVs_scaled[col] = 0.5

# Filter to valid team abbreviations only (drop blank/malformed rows)
valid_teams = ['ARI','ATL','BAL','BUF','CAR','CHI','CIN','CLE','DAL','DEN',
               'DET','GB','HOU','IND','JAX','KC','LA','LAC','LV','MIA','MIN',
               'NE','NO','NYG','NYJ','PHI','PIT','SEA','SF','TB','TEN','WAS']
currAVs_scaled = currAVs_scaled[currAVs_scaled['Team'].isin(valid_teams)].reset_index(drop=True)

print(currAVs_scaled.to_string())
currAVs_scaled


In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load AVbyPositionGroup for historical scaling (unchanged)
file_path = '../PickleFiles/AVbyPositionGroup.pkl'
av_by_position_group = pd.read_pickle(file_path)

columns_to_scale = ['oline', 'qb', 'rb', 'wrte', 'dst']

scaler = MinMaxScaler()
av_by_position_group_scaled = av_by_position_group.copy()
av_by_position_group_scaled[columns_to_scale] = scaler.fit_transform(av_by_position_group[columns_to_scale])

# Save — rename Team -> team to match what views.py expects
currAVs_final = currAVs_scaled.rename(columns={'Team': 'team'})
currAVs_final.to_pickle('../PickleFiles/currAVs.pkl')
av_by_position_group_scaled.to_pickle('../PickleFiles/AVbyPositionGroup.pkl')

print("Saved currAVs.pkl with", len(currAVs_final), "teams")
print(currAVs_final.to_string())
